## Batch Georeference a Subset of Photos

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm

from single_photo_workflow import (
    download_photo,
    get_image_metadata,
    calculate_horizontal_extent,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

### Prepare the subset and batch

In [ ]:
SUBSET_PATH = Path(
    "data/historic_airphoto_index/subsets/"
    "subset_20000_carver_37_38_40_51_57_63.gpkg"
)

LAYER = "photos"

# to process the next batch, change this to 100, then 200, etc. 
START = 0
BATCH_SIZE = 100

In [ ]:
gdf = gpd.read_file(SUBSET_PATH, layer=LAYER)

In [ ]:
# Use the GeoPackage name as the downloads folder
DOWNLOAD_DIR = Path("data/downloads") / SUBSET_PATH.stem
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Stable order for repeatable chunking
gdf = gdf.sort_values("cty_year_id").reset_index(drop=True)

In [ ]:
batch_gdf = gdf.iloc[START:START + BATCH_SIZE].copy()

print(f"Selected {len(batch_gdf)} records")
print(f"Downloads: {DOWNLOAD_DIR}")

### Download JPEG, read metadata, calculate extent

In [ ]:
results = []

for _, photo in tqdm(
    batch_gdf.iterrows(),
    total=len(batch_gdf),
    desc="Preparing photos"
):
    photo_id = photo["cty_year_id"]

    image_path = DOWNLOAD_DIR / f"{photo_id}_original.jpg"

    result = {
        "source_id": photo["source_id"],
        "cty_year_id": photo_id,
        "width_px": None,
        "dpi_x": None,
        "h_extent": None,
        "date_download": None,
        "error": None,
    }

    try:
        image_path = download_photo(
            url=photo["url"],
            output_path=image_path,
        )

        result["date_download"] = datetime.now(
            timezone.utc
        ).isoformat()

        metadata = get_image_metadata(image_path)

        result.update(metadata)

        result["h_extent"] = calculate_horizontal_extent(
            width_px=metadata["width_px"],
            dpi_x=metadata["dpi_x"],
            scale=photo["scale"],
        )

    except Exception as e:
        result["error"] = str(e)

    results.append(result)

### Inspect results

In [ ]:
results_df = pd.DataFrame(results)

display(results_df.head())

print("\nResults:")
print(f"Processed: {len(results_df)}")
print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Failed: {results_df['error'].notna().sum()}")

In [ ]:
display(
    results_df[
        ["cty_year_id", "width_px", "dpi_x", "h_extent", "error"]
    ]
)